[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_78_Shared_State_Blackboard.ipynb)

# Lesson 78 — Shared State & the Blackboard
### Phase 9 · Multi-Agent Orchestration · Lesson 2 of ~6

In **Lesson 77** you wired agents together with `Message` objects sent
*point-to-point*: the orchestrator hands a task to a worker, the critic hands
a critique back to the solver. That works when the topology is small and
fixed. It stops working when *many* agents must share an *evolving* body of
state.

**Today's one idea:**

> Point-to-point wiring is **O(N²)** — every new agent means rewiring the
> rest. A **blackboard** flips it: agents never talk to each other. They read
> from and write to **one shared, structured workspace**. Coordination becomes
> *data-centric*, not *message-centric*. The price of that freedom is a new
> hard problem — **consistency when several agents write at once** — and most
> of this lesson is about paying that price correctly.

| Lesson | Topic | Status |
|---|---|---|
| L77 | Topologies: orchestrator/worker, sequential, parallel, evaluator | ✅ done |
| **L78** | **Shared state & the blackboard (this lesson)** | **▶ today** |
| L79 | Routing & handoff — pick the right agent per request | next |
| L80 | Planning & decomposition — agents that plan their own steps | — |
| L81 | Reliability — partial failure, retries, timeouts across agents | — |
| L82 | **Phase 9 capstone** — ship a 4th OSS artifact (orchestration app) | — |

Everything runs **offline and deterministically** — no API key needed. The
concurrency demos use real threads, but are engineered (with a `Barrier`) to
make races *reproducible* so the lesson never flakes.

## 0 · Setup

One dependency (`rich`) for readable tables. No model calls today — the point
is coordination *plumbing*, and plumbing is best taught deterministically.

In [ ]:
!pip install rich -q

import threading, time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Any, Callable
from rich.console import Console
from rich.table import Table
from rich import print as rprint

console = Console()
BASE = "/content"          # Colab working dir; the orchestra/ package lands here
rprint("[green]OK[/] setup complete — no API key required for this lesson.")

## 1 · Why point-to-point wiring explodes

In L77 an agent talked to another agent *directly*. Suppose you have N agents
and they all need to see each other's outputs (a common situation once agents
collaborate on shared findings). How many communication channels is that?

* **Point-to-point mesh:** every pair needs a channel → `N·(N-1)/2` edges.
* **Blackboard hub:** every agent connects to *one* board → `N` edges.

The mesh grows **quadratically**; the hub grows **linearly**. That gap is the
entire architectural argument for a blackboard. Let's just count it.

In [ ]:
def mesh_edges(n):  return n * (n - 1) // 2   # everyone wires to everyone
def hub_edges(n):   return n                    # everyone wires to the board

tbl = Table(title="Communication channels: mesh vs blackboard")
tbl.add_column("agents (N)", justify="right")
tbl.add_column("point-to-point mesh", justify="right")
tbl.add_column("blackboard hub", justify="right")
tbl.add_column("mesh / hub", justify="right")
for n in (2, 3, 5, 10, 20):
    tbl.add_row(str(n), str(mesh_edges(n)), str(hub_edges(n)),
                f"{mesh_edges(n)/hub_edges(n):.1f}x")
console.print(tbl)

assert mesh_edges(20) == 190 and hub_edges(20) == 20
assert mesh_edges(20) / hub_edges(20) > 9          # nearly 10x wiring at N=20
rprint("[green]OK[/] At 20 agents the mesh needs [bold]190[/] channels; the "
       "blackboard needs [bold]20[/]. Add a 21st agent: mesh +20 edges, board +1.")

## 2 · The blackboard pattern

The metaphor is a team of specialists around a physical blackboard. Nobody
speaks to anyone; each expert **watches the board**, and when they see
something they can act on, they step up and **write their contribution**.
Eventually the board holds the solution. The idea is old — it powered the
**Hearsay-II** speech-understanding system in the 1970s — and it has three
parts:

1. **Knowledge sources (the agents):** each knows *when* it can contribute
   (a **trigger** / precondition) and *what* to write (its **run** step).
2. **The blackboard (shared state):** a structured workspace everyone reads
   and writes. This is the *only* thing agents touch.
3. **Control (the scheduler):** decides which knowledge source fires next.
   We'll use the simplest useful strategy — **data-driven**: run any source
   whose preconditions are now satisfied, repeat until nothing new fires.

Here's the smallest possible board and three agents collaborating **without a
single direct message between them**.

In [ ]:
# A deliberately naive board: a plain dict with read/write. We'll break it
# on purpose in the next section, then fix it.
class NaiveBoard:
    def __init__(self): self.data = {}
    def get(self, k, d=None): return self.data.get(k, d)
    def set(self, k, v): self.data[k] = v
    def has(self, k): return k in self.data

board = NaiveBoard()
board.set("question", "How do we cut LLM serving cost?")

# Three agents. Each only reads the board and writes back. They never call
# each other. Order below does NOT encode dependencies — the triggers do.
def researcher(b):
    if b.has("question") and not b.has("facts"):
        b.set("facts", ["cache identical prompts", "batch requests", "smaller model for easy asks"])

def writer(b):
    if b.has("facts") and not b.has("draft"):
        b.set("draft", "To cut cost: " + "; ".join(b.get("facts")) + ".")

def editor(b):
    if b.has("draft") and not b.has("final"):
        b.set("final", b.get("draft").replace("cut cost", "cut serving cost"))

# Naive control: just sweep them a few times until the board stops changing.
for _ in range(5):
    before = dict(board.data)
    for agent in (editor, writer, researcher):   # scrambled order on purpose
        agent(board)
    if board.data == before:
        break

rprint(f"[bold]facts :[/] {board.get('facts')}")
rprint(f"[bold]draft :[/] {board.get('draft')}")
rprint(f"[bold]final :[/] {board.get('final')}")
assert board.get("final") and "serving cost" in board.get("final")
rprint("[green]OK[/] Three agents solved it via shared state. Note the scrambled "
       "call order still worked — [italic]data readiness[/], not wiring, drove it.")

## 3 · The catch: concurrent writes

The naive board ran everything on **one thread**, one agent at a time. The
whole reason you reach for multiple agents (L77, Topology C) is to run them
**in parallel**. The moment two agents write the same key at the same time,
the naive board is *wrong*.

The classic failure is the **lost update**:

```
Agent A reads counter = 5
Agent B reads counter = 5      # both read the SAME value
Agent A writes 5 + 1 = 6
Agent B writes 5 + 1 = 6       # B clobbers A; the increment is LOST
```

Two increments happened; the counter only moved by one. Let's make it happen
on purpose — and *deterministically*, using a `threading.Barrier` so every
thread reads the old value before anyone writes.

In [ ]:
N_AGENTS = 8
board = NaiveBoard(); board.set("counter", 0)
barrier = threading.Barrier(N_AGENTS)   # forces all reads to precede all writes

def naive_increment():
    v = board.get("counter")   # 1. read
    barrier.wait()             #    ...everyone has now read the SAME value
    board.set("counter", v + 1)  # 2. write — clobbers the others

threads = [threading.Thread(target=naive_increment) for _ in range(N_AGENTS)]
for t in threads: t.start()
for t in threads: t.join()

got, expected = board.get("counter"), N_AGENTS
rprint(f"[bold]counter after {N_AGENTS} concurrent increments:[/] {got}  "
       f"(expected {expected})")
assert got == 1 and got < expected      # 7 of the 8 updates were lost
rprint(f"[red]LOST UPDATE:[/] {expected - got} of {expected} increments "
       "vanished. A plain dict is [bold]not[/] safe under parallel writers.")

## 4 · Making the board safe

Three standard fixes, in increasing sophistication:

| Fix | Idea | Trade-off |
|---|---|---|
| **Lock + atomic update** | hold a lock across the *whole* read-modify-write | simple, correct; the lock serializes writers (pessimistic) |
| **Compare-and-set (CAS)** | tag each key with a **version**; only write if the version is unchanged; retry on conflict | lock-free, scales better under low contention; you must handle retries |
| **Append-only log** | never mutate; append events, derive state by folding | full audit trail; readers do more work |

We'll build the first two into a proper `Blackboard` class. Every write bumps
a **version** per key — that version is what makes CAS possible.

In [ ]:
class Blackboard:
    # Thread-safe workspace with per-key versions.
    def __init__(self):
        self._data = {}; self._versions = {}; self._lock = threading.RLock()

    def get(self, k, d=None):
        with self._lock: return self._data.get(k, d)

    def version(self, k):
        with self._lock: return self._versions.get(k, 0)

    def get_versioned(self, k, d=None):
        with self._lock: return self._data.get(k, d), self._versions.get(k, 0)

    def set(self, k, v):                       # unconditional write
        with self._lock:
            self._data[k] = v
            self._versions[k] = self._versions.get(k, 0) + 1
            return self._versions[k]

    def update(self, k, fn, default=None):     # ATOMIC read-modify-write
        with self._lock:                        # lock held for the WHOLE op
            new = fn(self._data.get(k, default))
            self._data[k] = new
            self._versions[k] = self._versions.get(k, 0) + 1
            return new

    def compare_and_set(self, k, expected_version, v):   # OPTIMISTIC write
        with self._lock:
            if self._versions.get(k, 0) != expected_version:
                return False                    # someone wrote first -> caller retries
            self._data[k] = v
            self._versions[k] = expected_version + 1
            return True

    def has(self, k):
        with self._lock: return k in self._data
    def keys(self):
        with self._lock: return set(self._data)
    def snapshot(self):
        with self._lock: return dict(self._data)

rprint("[green]OK[/] Blackboard defined: get/set + [bold]update[/] (atomic) + "
       "[bold]compare_and_set[/] (versioned).")

### 4a · Pessimistic fix — atomic `update()`

`update(key, fn)` runs `fn(old) -> new` **with the lock held the entire time**,
so no other thread can slip between the read and the write. Same 8 concurrent
incrementers, same barrier forcing maximum contention — but now nothing is
lost.

In [ ]:
bb = Blackboard(); bb.set("counter", 0)
barrier = threading.Barrier(N_AGENTS)

def safe_increment():
    barrier.wait()                       # all start together = maximum contention
    bb.update("counter", lambda v: v + 1)  # but the whole RMW is atomic

threads = [threading.Thread(target=safe_increment) for _ in range(N_AGENTS)]
for t in threads: t.start()
for t in threads: t.join()

got = bb.get("counter")
rprint(f"[bold]counter with atomic update():[/] {got}  (expected {N_AGENTS})")
assert got == N_AGENTS                    # zero lost updates
rprint("[green]OK[/] The lock serialized the read-modify-write. All 8 increments survived.")

### 4b · Optimistic fix — compare-and-set with retry

A lock makes writers *wait*. When conflicts are rare, it's cheaper to be
**optimistic**: read the value *and its version*, compute the new value, then
try to write **only if the version hasn't changed**. If someone beat you, your
CAS fails — so you re-read and try again.

To make the conflict *deterministic*, every thread reads the version **before**
the barrier, so all 8 attempt their first CAS against the *same* version. Only
one can win each round; the losers retry. We count the retries to prove the
conflict was real.

In [ ]:
bb = Blackboard(); bb.set("counter", 0)
barrier = threading.Barrier(N_AGENTS)
retries = {"n": 0}; rlock = threading.Lock()

def cas_increment():
    val, ver = bb.get_versioned("counter")   # read BEFORE the barrier
    barrier.wait()                            # -> all 8 hold the same version
    while True:
        if bb.compare_and_set("counter", ver, val + 1):
            break                             # won the race
        with rlock: retries["n"] += 1         # lost -> count it and re-read
        val, ver = bb.get_versioned("counter")

threads = [threading.Thread(target=cas_increment) for _ in range(N_AGENTS)]
for t in threads: t.start()
for t in threads: t.join()

got = bb.get("counter")
rprint(f"[bold]counter with CAS:[/] {got}  (expected {N_AGENTS})   "
       f"[dim]retries: {retries['n']}[/]")
assert got == N_AGENTS                        # correct final value
assert retries["n"] > 0                       # conflicts really happened & were handled
rprint(f"[green]OK[/] CAS reached the right answer with no lock on the write path; "
       f"{retries['n']} conflicting attempts safely retried.")

## 5 · Control — who fires next?

A safe board still needs a **scheduler**. The blackboard's signature strategy
is **data-driven control**: a knowledge source declares a **trigger**
(a precondition over the board) and a **run** step. The controller loops,
firing every source whose trigger is satisfied, until a full pass fires
*nothing* — a state called **quiescence**. That fixed point is the answer.

Two things this buys you:
* **No hard-coded order.** You never say "run the summarizer after the
  fetcher." You say *the summarizer needs `raw` on the board*. Ordering
  emerges from data readiness.
* **A termination guarantee** — as long as sources only fire when they add
  something new (their trigger includes "...and my output isn't there yet"),
  the board reaches quiescence. `max_rounds` guards against a mistake.

Let's build a small research-brief pipeline entirely by writing to the board.

In [ ]:
@dataclass
class KnowledgeSource:
    name: str
    trigger: Callable          # (bb) -> bool : may I fire?
    run: Callable              # (bb) -> None : do my work, write to bb

def run_until_quiescent(bb, sources, max_rounds=100):
    rounds, fired = 0, []
    for _ in range(max_rounds):
        rounds += 1; fired_this_round = False
        for ks in sources:
            if ks.trigger(bb):
                ks.run(bb); fired.append(ks.name); fired_this_round = True
        if not fired_this_round:
            break                          # quiescence: nothing new to do
    return rounds, fired

# A tiny deterministic "knowledge base" the fetcher reads from.
KB = {"LLM serving cost": "Cache prompts, batch requests, right-size the model."}

bb = Blackboard(); bb.set("topic", "LLM serving cost")

sources = [
    KnowledgeSource("fetch",
        lambda b: b.has("topic") and not b.has("raw"),
        lambda b: b.set("raw", KB.get(b.get("topic"), "no data"))),
    KnowledgeSource("summarize",
        lambda b: b.has("raw") and not b.has("summary"),
        lambda b: b.set("summary", b.get("raw").split(".")[0] + ".")),
    KnowledgeSource("critique",
        lambda b: b.has("summary") and not b.has("critique"),
        lambda b: b.set("critique", "specific and actionable")),
    KnowledgeSource("finalize",
        lambda b: b.has("summary") and b.has("critique") and not b.has("final"),
        lambda b: b.set("final", f"{b.get('summary')} (review: {b.get('critique')})")),
]

# Register them in a DELIBERATELY WRONG order to prove order doesn't matter.
import random; random.seed(78); shuffled = sources[:]; random.shuffle(shuffled)
rprint(f"[dim]registration order: {[s.name for s in shuffled]}[/]")

rounds, fired = run_until_quiescent(bb, shuffled)
tbl = Table(title="Blackboard after the run")
tbl.add_column("key"); tbl.add_column("value")
for k in ("topic", "raw", "summary", "critique", "final"):
    tbl.add_row(k, str(bb.get(k)))
console.print(tbl)
rprint(f"[bold]fire sequence:[/] {fired}    [bold]rounds:[/] {rounds}")

assert bb.has("final") and "review" in bb.get("final")
assert fired.index("fetch") < fired.index("summarize") < fired.index("finalize")
rprint("[green]OK[/] Despite scrambled registration, data-driven control fired "
       "fetch -> summarize -> ... -> finalize in the only order the data allowed.")

## 6 · The payoff — add an agent with **zero rewiring**

Here's the concrete win over point-to-point. To add a *translator* that
produces a Spanish version of the final brief, you do **not** touch the
controller, and you do **not** touch any existing agent. You append one
knowledge source that declares *"I fire when `final` exists and `final_es`
doesn't."* In a point-to-point mesh you'd have to find whoever produces
`final`, edit it to also call the translator, and update the wiring.

In [ ]:
GLOSSARY = {"LLM serving cost": "coste de servir LLMs"}

translator = KnowledgeSource("translate",
    lambda b: b.has("final") and not b.has("final_es"),
    lambda b: b.set("final_es",
                    b.get("final").replace("LLM serving cost", "")  # (demo stub)
                    or "Resumen: " + GLOSSARY.get(b.get("topic"), b.get("topic"))))

before_keys = bb.keys()
extended = shuffled + [translator]        # the ONLY change: one append
rounds2, fired2 = run_until_quiescent(bb, extended)

rprint(f"[bold]new key added:[/] final_es = {bb.get('final_es')!r}")
rprint(f"[bold]this run fired:[/] {fired2}   (only the new source had work to do)")

assert bb.has("final_es")                            # the new agent contributed
assert bb.keys() - before_keys == {"final_es"}       # nothing else changed
assert fired2 == ["translate"]                       # controller untouched, others quiescent
rprint("[green]OK[/] One append, no rewiring, no controller edits. That linear-cost "
       "extensibility is the whole reason the blackboard scales where a mesh doesn't.")

## 7 · Ship it — `orchestra/blackboard.py`

We add today's work to the `orchestra/` package started in L77. The module
source is embedded as **base64** so no quote or docstring inside it can ever
collide with this cell's delimiters (the collision-proof pattern from L77).
We write it to disk, import it back, and smoke-test the three guarantees:
atomic update, CAS-with-version, and data-driven control to quiescence.

In [ ]:
# === write orchestra/blackboard.py, then import & smoke-test it ===
import base64, importlib, sys
from pathlib import Path

_B64 = "IiIib3JjaGVzdHJhLmJsYWNrYm9hcmQg4oCUIGEgc2hhcmVkIHN0cnVjdHVyZWQgd29ya3NwYWNlIGZvciBtYW55IGFnZW50cy4KCkluIG9yY2hlc3RyYS5jb3JlIChMZXNzb24gNzcpIGFnZW50cyBwYXNzIE1lc3NhZ2VzIHBvaW50LXRvLXBvaW50LiBUaGF0IGlzCmZpbmUgZm9yIGEgZml4ZWQgaGFuZGZ1bCBvZiBhZ2VudHMuIFdoZW4gbWFueSBhZ2VudHMgbXVzdCBzaGFyZSBldm9sdmluZwpzdGF0ZSwgcG9pbnQtdG8tcG9pbnQgd2lyaW5nIGlzIE8oTl4yKTogZXZlcnkgbmV3IGFnZW50IG1lYW5zIHJld2lyaW5nIHRoZQpyZXN0LiBUaGUgYmxhY2tib2FyZCBmbGlwcyBpdDogYWdlbnRzIG5ldmVyIHRhbGsgdG8gZWFjaCBvdGhlci4gVGhleSByZWFkCmZyb20gYW5kIHdyaXRlIHRvIE9ORSBzaGFyZWQsIHN0cnVjdHVyZWQsIGNvbmN1cnJlbmN5LXNhZmUgd29ya3NwYWNlLiBUaGUKbmV3IGhhcmQgcHJvYmxlbSB0aGlzIGNyZWF0ZXMgaXMgQ09OU0lTVEVOQ1kgdW5kZXIgY29uY3VycmVudCB3cml0ZXMsIHdoaWNoCmlzIHdoeSBCbGFja2JvYXJkIGlzIGJ1aWx0IGFyb3VuZCB2ZXJzaW9uZWQsIGF0b21pYyBvcGVyYXRpb25zLgoiIiIKaW1wb3J0IHRocmVhZGluZwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIENhbGxhYmxlLCBPcHRpb25hbAoKCmNsYXNzIEJsYWNrYm9hcmQ6CiAgICAiIiJBIHRocmVhZC1zYWZlIGtleS92YWx1ZSB3b3Jrc3BhY2Ugd2l0aCBwZXIta2V5IHZlcnNpb25zLgoKICAgIEV2ZXJ5IHN1Y2Nlc3NmdWwgd3JpdGUgYnVtcHMgdGhhdCBrZXkncyB2ZXJzaW9uLiBWZXJzaW9ucyBhcmUgd2hhdCBtYWtlCiAgICBvcHRpbWlzdGljIGNvbmN1cnJlbmN5IChjb21wYXJlLWFuZC1zZXQpIHBvc3NpYmxlOiBhbiBhZ2VudCBjYW4gcHJvdmUgaXQKICAgIGlzIHdyaXRpbmcgb24gdG9wIG9mIHRoZSBzdGF0ZSBpdCBhY3R1YWxseSByZWFkLCBub3QgYSBzdGFsZSBjb3B5LgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYuX2RhdGEgPSB7fQogICAgICAgIHNlbGYuX3ZlcnNpb25zID0ge30KICAgICAgICBzZWxmLl9sb2NrID0gdGhyZWFkaW5nLlJMb2NrKCkKCiAgICBkZWYgZ2V0KHNlbGYsIGtleSwgZGVmYXVsdD1Ob25lKToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9kYXRhLmdldChrZXksIGRlZmF1bHQpCgogICAgZGVmIHZlcnNpb24oc2VsZiwga2V5KToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl92ZXJzaW9ucy5nZXQoa2V5LCAwKQoKICAgIGRlZiBnZXRfdmVyc2lvbmVkKHNlbGYsIGtleSwgZGVmYXVsdD1Ob25lKToKICAgICAgICAiIiJSZXR1cm4gKHZhbHVlLCB2ZXJzaW9uKSBhdG9taWNhbGx5IOKAlCB0aGUgcGFpciBDQVMgbmVlZHMuIiIiCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZGF0YS5nZXQoa2V5LCBkZWZhdWx0KSwgc2VsZi5fdmVyc2lvbnMuZ2V0KGtleSwgMCkKCiAgICBkZWYgc2V0KHNlbGYsIGtleSwgdmFsdWUpOgogICAgICAgICIiIlVuY29uZGl0aW9uYWwgd3JpdGUuIEJ1bXBzIHRoZSB2ZXJzaW9uLiBOT1Qgc2FmZSBmb3IKICAgICAgICByZWFkLW1vZGlmeS13cml0ZSByYWNlcyBvbiBpdHMgb3duIOKAlCB0aGF0IGlzIHdoYXQgdXBkYXRlL0NBUyBmaXguIiIiCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzZWxmLl9kYXRhW2tleV0gPSB2YWx1ZQogICAgICAgICAgICBzZWxmLl92ZXJzaW9uc1trZXldID0gc2VsZi5fdmVyc2lvbnMuZ2V0KGtleSwgMCkgKyAxCiAgICAgICAgICAgIHJldHVybiBzZWxmLl92ZXJzaW9uc1trZXldCgogICAgZGVmIHVwZGF0ZShzZWxmLCBrZXksIGZuLCBkZWZhdWx0PU5vbmUpOgogICAgICAgICIiIkF0b21pYyByZWFkLW1vZGlmeS13cml0ZTogZm4ob2xkX3ZhbHVlKSAtPiBuZXdfdmFsdWUsIGV4ZWN1dGVkCiAgICAgICAgd2l0aCB0aGUgbG9jayBoZWxkIGZvciB0aGUgV0hPTEUgb3BlcmF0aW9uLiBUaGlzIGlzIHRoZSBwZXNzaW1pc3RpYwogICAgICAgIGZpeCBmb3IgbG9zdCB1cGRhdGVzLiIiIgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgb2xkID0gc2VsZi5fZGF0YS5nZXQoa2V5LCBkZWZhdWx0KQogICAgICAgICAgICBuZXcgPSBmbihvbGQpCiAgICAgICAgICAgIHNlbGYuX2RhdGFba2V5XSA9IG5ldwogICAgICAgICAgICBzZWxmLl92ZXJzaW9uc1trZXldID0gc2VsZi5fdmVyc2lvbnMuZ2V0KGtleSwgMCkgKyAxCiAgICAgICAgICAgIHJldHVybiBuZXcKCiAgICBkZWYgY29tcGFyZV9hbmRfc2V0KHNlbGYsIGtleSwgZXhwZWN0ZWRfdmVyc2lvbiwgdmFsdWUpOgogICAgICAgICIiIk9wdGltaXN0aWMgd3JpdGU6IG9ubHkgc3VjY2VlZHMgaWYgdGhlIGtleSdzIHZlcnNpb24gc3RpbGwgZXF1YWxzCiAgICAgICAgZXhwZWN0ZWRfdmVyc2lvbi4gUmV0dXJucyBUcnVlIG9uIHN1Y2Nlc3MsIEZhbHNlIGlmIHNvbWVvbmUgZWxzZSB3cm90ZQogICAgICAgIGZpcnN0ICh0aGUgY2FsbGVyIHNob3VsZCByZS1yZWFkIGFuZCByZXRyeSkuIiIiCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBpZiBzZWxmLl92ZXJzaW9ucy5nZXQoa2V5LCAwKSAhPSBleHBlY3RlZF92ZXJzaW9uOgogICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgICAgIHNlbGYuX2RhdGFba2V5XSA9IHZhbHVlCiAgICAgICAgICAgIHNlbGYuX3ZlcnNpb25zW2tleV0gPSBleHBlY3RlZF92ZXJzaW9uICsgMQogICAgICAgICAgICByZXR1cm4gVHJ1ZQoKICAgIGRlZiBoYXMoc2VsZiwga2V5KToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiBrZXkgaW4gc2VsZi5fZGF0YQoKICAgIGRlZiBrZXlzKHNlbGYpOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgcmV0dXJuIHNldChzZWxmLl9kYXRhLmtleXMoKSkKCiAgICBkZWYgc25hcHNob3Qoc2VsZik6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICByZXR1cm4gZGljdChzZWxmLl9kYXRhKQoKCkBkYXRhY2xhc3MKY2xhc3MgS25vd2xlZGdlU291cmNlOgogICAgIiIiQSBrbm93bGVkZ2Ugc291cmNlIChhZ2VudCkgZGVjbGFyZXMgV0hFTiBpdCBtYXkgZmlyZSAodHJpZ2dlcikgYW5kCiAgICBXSEFUIGl0IGRvZXMgKHJ1bikuIEl0IHJlYWRzL3dyaXRlcyBvbmx5IHRoZSBibGFja2JvYXJkIOKAlCBuZXZlciBhbm90aGVyCiAgICBhZ2VudCBkaXJlY3RseS4gQ29udHJvbCBpcyBkYXRhLWRyaXZlbjogYSBzb3VyY2UgZmlyZXMgd2hlbiBpdHMKICAgIHByZWNvbmRpdGlvbnMgYXBwZWFyIG9uIHRoZSBib2FyZC4iIiIKICAgIG5hbWU6IHN0cgogICAgdHJpZ2dlcjogQ2FsbGFibGVbW0JsYWNrYm9hcmRdLCBib29sXQogICAgcnVuOiBDYWxsYWJsZVtbQmxhY2tib2FyZF0sIE5vbmVdCgoKZGVmIHJ1bl91bnRpbF9xdWllc2NlbnQoYmIsIHNvdXJjZXMsIG1heF9yb3VuZHM9MTAwKToKICAgICIiIkRhdGEtZHJpdmVuIGNvbnRyb2xsZXI6IHJlcGVhdGVkbHkgcnVuIGV2ZXJ5IHNvdXJjZSB3aG9zZSB0cmlnZ2VyCiAgICBmaXJlcywgdW50aWwgYSBmdWxsIHBhc3MgZmlyZXMgbm90aGluZyAocXVpZXNjZW5jZSkgb3IgbWF4X3JvdW5kcyBpcyBoaXQuCiAgICBSZXR1cm5zIChyb3VuZHMsIGZpcmVkKSB3aGVyZSBmaXJlZCBpcyB0aGUgb3JkZXJlZCBsaXN0IG9mIHNvdXJjZSBuYW1lcwogICAgdGhhdCByYW4uIG1heF9yb3VuZHMgaXMgdGhlIGd1YXJkIGFnYWluc3QgYSBub24tdGVybWluYXRpbmcgYm9hcmQuIiIiCiAgICByb3VuZHMgPSAwCiAgICBmaXJlZCA9IFtdCiAgICBmb3IgXyBpbiByYW5nZShtYXhfcm91bmRzKToKICAgICAgICByb3VuZHMgKz0gMQogICAgICAgIGZpcmVkX3RoaXNfcm91bmQgPSBGYWxzZQogICAgICAgIGZvciBrcyBpbiBzb3VyY2VzOgogICAgICAgICAgICBpZiBrcy50cmlnZ2VyKGJiKToKICAgICAgICAgICAgICAgIGtzLnJ1bihiYikKICAgICAgICAgICAgICAgIGZpcmVkLmFwcGVuZChrcy5uYW1lKQogICAgICAgICAgICAgICAgZmlyZWRfdGhpc19yb3VuZCA9IFRydWUKICAgICAgICBpZiBub3QgZmlyZWRfdGhpc19yb3VuZDoKICAgICAgICAgICAgYnJlYWsKICAgIHJldHVybiByb3VuZHMsIGZpcmVkCg=="

pkg = Path(BASE) / "orchestra"
pkg.mkdir(parents=True, exist_ok=True)
# ensure it's a package (L77 created __init__.py in the Colab runtime; be safe)
init = pkg / "__init__.py"
if not init.exists():
    init.write_text("")
(pkg / "blackboard.py").write_bytes(base64.b64decode(_B64))

if BASE not in sys.path:
    sys.path.insert(0, BASE)
importlib.invalidate_caches()
import orchestra.blackboard as bbmod
importlib.reload(bbmod)
from orchestra.blackboard import Blackboard as BB, KnowledgeSource as KS, run_until_quiescent as ruq

# smoke test 1: atomic update under contention
b = BB(); b.set("n", 0); bar = threading.Barrier(6)
def _inc():
    bar.wait(); b.update("n", lambda v: v + 1)
ts = [threading.Thread(target=_inc) for _ in range(6)]
[t.start() for t in ts]; [t.join() for t in ts]
assert b.get("n") == 6

# smoke test 2: compare_and_set respects versions
b2 = BB(); b2.set("x", "a"); _, v = b2.get_versioned("x")
assert b2.compare_and_set("x", v, "b") is True         # version matches -> writes
assert b2.compare_and_set("x", v, "c") is False        # stale version -> refused
assert b2.get("x") == "b"

# smoke test 3: data-driven control reaches quiescence
b3 = BB(); b3.set("seed", 1)
srcs = [KS("double", lambda z: z.has("seed") and not z.has("out"),
           lambda z: z.set("out", z.get("seed") * 2))]
r, f = ruq(b3, srcs)
assert b3.get("out") == 2 and f == ["double"]

rprint("[green]OK[/] orchestra/blackboard.py written, imported, and all 3 "
       "guarantees smoke-tested. The package now has core.py (L77) + blackboard.py (L78).")

## 8 · Ten ways a blackboard goes wrong

| # | Pitfall | Fix |
|---|---|---|
| 1 | Board as a global mutable *god object* everyone dumps into | keep it **structured** — named keys with owners/schema, not a junk drawer |
| 2 | No concurrency control under parallel agents | `update()` (lock) or `compare_and_set` (versions) — never a bare dict |
| 3 | Lost updates from read-modify-write races | make the whole RMW atomic; never read-then-set across a gap |
| 4 | Unbounded board growth (every step appends forever) | cap history / evict stale keys / summarize |
| 5 | No quiescence check → controller loops forever | triggers must include "…and my output isn't there yet"; keep `max_rounds` |
| 6 | Two sources fight over the same key (thrashing) | give each output a single writer, or use CAS + a merge rule |
| 7 | Reading a stale version and acting on it | read `(value, version)` together; re-read on CAS failure |
| 8 | Lock held during a slow LLM call (serializes everyone) | do the slow work *outside* the lock; only the final write is atomic |
| 9 | No provenance — you can't tell who wrote what | stamp writes with agent name + timestamp (append-only log variant) |
| 10 | Blackboard becomes the bottleneck / single point of failure | shard by namespace, or fall back to point-to-point for hot paths |

## 9 · Verification — every claim, re-checked

In [ ]:
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    rprint(("[green]PASS[/] " if cond else "[red]FAIL[/] ") + name)

check("mesh is O(N^2): 20 agents -> 190 edges", mesh_edges(20) == 190)
check("hub is O(N): 20 agents -> 20 edges", hub_edges(20) == 20)

# re-run the core demos freshly for the checklist
from orchestra.blackboard import Blackboard as BB2
b = BB2(); b.set("c", 0); bar = threading.Barrier(8)
def _lost():
    v = b.get("c"); bar.wait(); b.set("c", v + 1)
ts = [threading.Thread(target=_lost) for _ in range(8)]; [t.start() for t in ts]; [t.join() for t in ts]
check("barrier forces a deterministic lost update (==1)", b.get("c") == 1)

b = BB2(); b.set("c", 0); bar = threading.Barrier(8)
def _safe():
    bar.wait(); b.update("c", lambda v: v + 1)
ts = [threading.Thread(target=_safe) for _ in range(8)]; [t.start() for t in ts]; [t.join() for t in ts]
check("atomic update() has zero lost updates (==8)", b.get("c") == 8)

b = BB2(); b.set("c", 0)
_, v0 = b.get_versioned("c")
check("CAS succeeds on current version", b.compare_and_set("c", v0, 1) is True)
check("CAS refuses a stale version", b.compare_and_set("c", v0, 99) is False)

check("data-driven run reached final brief", bb.has("final"))
check("adding a 5th agent changed only its own key", bb.keys().issuperset({"final_es"}))
check("orchestra.blackboard imports the shipped module", "orchestra.blackboard" in sys.modules)

passed = sum(ok for _, ok in checks)
rprint(f"\n[bold]{passed}/{len(checks)} checks passed[/]")
assert passed == len(checks), "some checks failed"
rprint("[bold green]ALL CHECKS PASSED ✅[/]")

## 10 · Summary & homework

**What you built today**

| Concept | One-liner |
|---|---|
| N² wiring problem | point-to-point channels grow quadratically; a hub grows linearly |
| Blackboard | one shared structured workspace; agents read/write it, never each other |
| Lost update | concurrent read-modify-write silently drops writes on a bare dict |
| Atomic `update()` | lock held across the whole RMW — pessimistic, simple, correct |
| Compare-and-set | per-key versions + retry — optimistic, lock-free write path |
| Data-driven control | triggers fire on data readiness; loop to **quiescence** |
| Linear extensibility | add an agent = append one knowledge source, zero rewiring |

**Homework (extend `orchestra/blackboard.py`):**

1. **Provenance.** Make every write record `(agent, timestamp)`. Add
   `history(key)` returning the ordered list of writers. (Pitfall #9.)
2. **Append-only variant.** Add an `EventLog` board that never mutates —
   `append(event)` + `fold(reducer)` to derive current state. Compare its
   audit story to the mutable board's.
3. **Slow work outside the lock.** Simulate a 100 ms "LLM call" inside a
   knowledge source. Show that doing it *inside* `update()` serializes all
   agents, but doing it *before* a short atomic write keeps them parallel.
   Measure both with `ThreadPoolExecutor`. (Pitfall #8.)
4. **Priority control.** Replace `run_until_quiescent`'s flat sweep with a
   controller that, when several sources trigger, runs the highest-priority
   one first. Show it changes the fire sequence but not the final board.
5. **Wire it to L77.** Have an `orchestra.core` `Agent` write its result to a
   `Blackboard` instead of returning a `Message`, so a supervisor and its
   workers coordinate through shared state. This is the bridge into L79.

**Next lesson — L79: Routing & Handoff.** Today many agents shared *one*
board. Next: how does a request find the *right* agent in the first place? A
router/dispatcher classifies each incoming task and hands it to the best
specialist (or escalates when unsure) — the multi-agent equivalent of a load
balancer with judgment.

*You now own `orchestra/` with two modules: `core.py` (topologies, L77) and
`blackboard.py` (shared state, L78), heading toward the L82 Phase-9 capstone.*